## **[ Goal ]**: Create a hypothetical dataset with a simple linear relationship y = 2x + 1 and implement a model that learns this relationship using all of PyTorch's high-level APIs.

### Step 1: Importing Required Libraries and Creating Fake Data
- First, import PyTorch and NumPy. Then, create fake data that follows the relationship y = 2x + 1. 
- It's recommended to add some noise (np.random.randn) to mimic real data.

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import numpy as np

# Create a virtual dataset
# Data with the relationship y = 2x + 1
X_numpy = np.arange(1, 101, 1, dtype=np.float32).reshape(-1, 1)
y_numpy = 2 * X_numpy + 1 + np.random.randn(100, 1).astype(np.float32) * 5 # Add some noise

# Convert NumPy arrays to PyTorch tensors
X_train = torch.from_numpy(X_numpy)
y_train = torch.from_numpy(y_numpy)

print(f"First 5 samples of X_train: \n{X_train[:5]}")
print(f"First 5 samples of y_train: \n{y_train[:5]}")

### Step 2: Define the Dataset Class

- Define our own dataset class by implementing __len__ and __getitem__. 
- __getitem__ returns the (data, answer) pair corresponding to idx.

In [ ]:
class MyLinearDataset(Dataset):
    def __init__(self, x_tensor, y_tensor):
        self.x = x_tensor
        self.y = y_tensor

    def __getitem__(self, index):
        return (self.x[index], self.y[index])

    def __len__(self):
        return len(self.x)

# Create Dataset object
train_dataset = MyLinearDataset(X_train, y_train)

### Step 3: Prepare data with DataLoader
- Pass the Dataset object to DataLoader. 
- Prepare the data to be used for learning by setting the batch_size and shuffle options.

In [ ]:
# Create DataLoader object
batch_size = 10
train_loader = DataLoader(dataset=train_dataset,
                          batch_size=batch_size,
                          shuffle=True) # Turn on shuffle option to randomly shuffle data

### Step 4: Design a model with nn.Module
- Define a linear model that approximates y=wx+b by inheriting from nn.Module. 
- Use an nn.Linear layer that receives 1 input feature (in_features=1) and predicts 1 output value (out_features=1).

In [ ]:
class LinearRegressionModel(nn.Module):
    def __init__(self):
        super(LinearRegressionModel, self).__init__()
        # Define a linear layer with 1 input and 1 output
        self.linear = nn.Linear(in_features=1, out_features=1)

    def forward(self, x):
        # Pass the defined layer to return the predicted value
        return self.linear(x)

# Create model object
model = LinearRegressionModel()
print(model)

### Step 5: Set up the loss function and optimizer
- Since it is a regression problem, use nn.MSELoss as the loss function. 
- Use the most basic optim.SGD as the optimizer, and pass model.
- parameters() to set it to manage the model's parameters.

In [ ]:
# Define loss function
criterion = nn.MSELoss()

# Define optimizer
learning_rate = 0.01
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

### Step 6: Implement a complete learning loop (Training Loop)
- Now put all the pieces together and proceed with actual learning. 
- Repeat in epoch and batch units by nesting for statements. 
- Inside the loop, the 5 steps of gradient initialization -> forward propagation -> loss calculation -> back propagation -> parameter update are repeated continuously (actually 3 steps: zero_grad, backward, step including forward propagation and loss calculation).

In [ ]:
# Set learning parameters
num_epochs = 50

# Start learning loop
for epoch in range(num_epochs):
    # Get mini-batch one by one from DataLoader
    for i, (inputs, labels) in enumerate(train_loader):
        # 1. Forward pass
        outputs = model(inputs)
        loss = criterion(outputs, labels)

        # 2. Zero gradients
        optimizer.zero_grad()

        # 3. Backward pass
        loss.backward()

        # 4. Update weights
        optimizer.step()

    # Print logs every 10 epochs
    if (epoch+1) % 10 == 0:
        print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}')

print("\nLearning completed!")

### Step 7: Check the learning results
- Check how close the parameters of the learned model are to w=2 and b=1, which we initially set.

In [ ]:
# Print learned parameters
# model.parameters() is a generator, so convert it to a list to check
learned_params = list(model.parameters())
learned_w = learned_params[0].item()
learned_b = learned_params[1].item()

print(f"Learned weight (w): {learned_w:.4f}")
print(f"Learned bias (b): {learned_b:.4f}")